# Cat RAG Chatbot

This notebook demonstrates a Retrieval-Augmented Generation (RAG)
pipeline built on a domain-specific PDF about cats.

The goal is to reduce hallucinations by grounding LLM responses in
retrieved document context.

In [ ]:
!pip install -U \
  langchain \
  langchain-community \
  langchain-openai \
  faiss-cpu \
  pypdf \
  openai \
  tiktoken
!pip install -U langchain-text-splitters
!pip install -U sentence-transformers
!pip install -U langchain-huggingface

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings


In [3]:
loader = PyPDFLoader("cats.pdf")
documents = loader.load()

print(len(documents))


1


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)

chunks = splitter.split_documents(documents)
print(len(chunks))


4


In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


In [6]:
vectorstore = FAISS.from_documents(chunks, embeddings)


In [15]:
docs = vectorstore.similarity_search("Why do cats purr?", k=2)
for d in docs:
    print(d.page_content[:80])


purr when they are anxious, injured, or seeking comfort.
Cats communicate throug
such as taurine, vitamin A, and arachidonic acid, which are found in animal tiss


## Notes

- Free sentence-transformer embeddings were used to avoid API limits
- FAISS enables fast semantic search over document chunks
- The pipeline is fully reproducible from this repository
